# Merge EPA Water Quality/Climate dataset with Agriculture

In [1]:
import pandas as pd
import numpy as np

# Load EPA-Climate merged data
epa_climate_df = pd.read_csv('../../data/tabular/merged/epa-climate-merged.csv')

# Load Agriculture data
ag_df = pd.read_csv('../../data/tabular/agricultural/clean/usdaNass-agriculture-clean.csv')

In [2]:
print("Preparing data for merge...")
# Ensure date columns are datetime
epa_climate_df['day_dt'] = pd.to_datetime(epa_climate_df['day'])
epa_climate_df['year'] = epa_climate_df['day_dt'].dt.year
epa_climate_df['state'] = 'IOWA' 

print("Pivoting agriculture data to wide format...")
# Clean up Data Item names for column headers
ag_df['data_item_short'] = ag_df['data_item'].str.replace(', MEASURED IN .*', '', regex=True)

# Pivot
ag_pivot = ag_df.pivot_table(
    index=['year', 'period', 'state'],
    columns='data_item_short',
    values='value',
    aggfunc='first'
).reset_index()

# Create month mapping for Ag data
months = ['JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC']
ag_pivot['month'] = ag_pivot['period'].where(ag_pivot['period'].isin(months))

# For dates in epa_climate, get the month
epa_climate_df['month'] = epa_climate_df['day_dt'].dt.strftime('%b').str.upper()

Preparing data for merge...
Pivoting agriculture data to wide format...


In [3]:
print("Merging on Year, Month, and State...")
merged_df = epa_climate_df.merge(
    ag_pivot[ag_pivot['month'].notna()],
    left_on=['year', 'month', 'state'],
    right_on=['year', 'month', 'state'],
    how='left'
)

print(f"Final merged shape: {merged_df.shape}")

Merging on Year, Month, and State...
Final merged shape: (59763, 3845)


In [4]:
merged_df.head(5)

,ActivityIdentifier,MonitoringLocationIdentifier,CharacteristicName,ResultMeasureValue,ResultMeasure/MeasureUnitCode,ResultValueTypeName,ActivityStartDateTime,OrganizationIdentifier,MonitoringLocationName,MonitoringLocationTypeName,...,"WOODY ORNAMENTALS & VINES, OTHER, RETAIL - OPERATIONS WITH SALES","WOODY ORNAMENTALS & VINES, OTHER, VINCA - OPERATIONS WITH INVENTORY","WOODY ORNAMENTALS & VINES, OTHER, VINCA - OPERATIONS WITH SALES","WOODY ORNAMENTALS & VINES, OTHER, VINCA - SALES","WOODY ORNAMENTALS & VINES, OTHER, VINCA, RETAIL - OPERATIONS WITH SALES","WOODY ORNAMENTALS & VINES, OTHER, VINCA, WHOLESALE - OPERATIONS WITH SALES","WOODY ORNAMENTALS & VINES, OTHER, WHOLESALE - OPERATIONS WITH SALES",WOOL - PRICE RECEIVED,WOOL - PRODUCTION,WOOL - SHORN
0,nwisia.01.02400517,USGS-05474000,"Temperature, water",0.0,deg C,Actual,2024-01-01 12:00:00,USGS-IA,"Skunk River at Augusta, IA",Stream,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,nwisia.01.02400551,USGS-05451210,Stream width measure,30.0,ft,Actual,2024-02-26 10:30:00,USGS-IA,"South Fork Iowa River NE of New Providence, IA",Stream,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,nwisia.01.02400551,USGS-05451210,"Temperature, water",4.7,deg C,Actual,2024-02-26 10:30:00,USGS-IA,"South Fork Iowa River NE of New Providence, IA",Stream,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,nwisia.01.02400551,USGS-05451210,"Temperature, air",8.9,deg C,Actual,2024-02-26 10:30:00,USGS-IA,"South Fork Iowa River NE of New Providence, IA",Stream,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,nwisia.01.02400551,USGS-05451210,Specific conductance,579.0,uS/cm @25C,Actual,2024-02-26 10:30:00,USGS-IA,"South Fork Iowa River NE of New Providence, IA",Stream,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Save the result (this may take a while due to the size of the data)
output_path = '../../data/tabular/merged/final-data.csv'
merged_df.to_csv(output_path, index=False)
print(f"Saved merged data to {output_path}")

Saved merged data to ../../data/tabular/merged/final-data.csv
